# Week 4 Project

- A code tool that automatically adds docstring/comments
- Also suggests test cases

In [ ]:
# import the necessary libraries
import gradio as gr
import requests

OLLAMA_MODEL = "phi3" # define the ollama model to use

In [ ]:
# Define a function to generate text using the Olla model
def run_ollama(prompt, model=OLLAMA_MODEL):
    # Define the URL where the API is hosted
    url = "http://localhost:11434/api/generate"

    # Prepare a dictionary containing the data to be sent to the API
    data = {
        "model": model, # Use the Olla model by default
        "prompt": prompt, # Use the prompt provided by the user
        "stream": False # Do not stream the response
    }
    try:
        # Send the data to the API and get the response
        r = requests.post(url, json=data, timeout=60)
        # Raise an exception if the response is not successful
        r.raise_for_status()
        # Return the response as a string   
        return r.json()["response"].strip()
    except Exception as e:
        # Return an error message if the response is not successful
        return f"ERROR: {str(e)}"

In [ ]:
# Define a function to automatically add docstring/comments and test cases
def auto_comment_and_test(code):
    comment_prompt = (
        "You are a helpful code assistant. Add clear, Pythonic comments line-by-line or for each block so "
        "a novice can easily understand. Do not change code unless strictly necessary.\n\n"
        "Python code:\n"
        f"{code}\n\n"
        "Commented code:\n"
    )
    # Run the Olla model to add docstring/comments  
    commented_code = run_ollama(comment_prompt)
    # If the response is not successful, return an error message
    if not commented_code or commented_code.startswith("ERROR"):
        commented_code = "Error: No output. Try changing your prompt/model."
    
    test_prompt = (
        "You are a helpful code assistant. Given the following Python function, create two minimal, clear test cases "
        "that would test its output. Use assert or simple print-based checks.\n"
        "Python code:\n"
        f"{code}\n\n"
        "Test cases:\n"
    )
    # Run the Olla model to add test cases
    test_cases = run_ollama(test_prompt)
    # If the response is not successful, return an error message
    if not test_cases or test_cases.startswith("ERROR"):
        test_cases = "Error: No output. Try changing your prompt/model."

    return commented_code, test_cases

In [ ]:
# Define the UI
with gr.Blocks() as demo:
    gr.Markdown("# 📝 Local LLM Code Commenter & Test Generator (Ollama Version)\n"
                "- Uses Ollama (REST API) and an open-source local LLM.\n"
                "- Paste a simple Python function below.\n"
                "- Returns commented code and suggested test cases."
                )
    code_in = gr.Code(label="Paste Python function", language="python")
    with gr.Row():
        b = gr.Button("Generate Comments & Tests", variant="primary")
    commented = gr.Code(label="Code with Comments", language="python")
    tests = gr.Code(label="Suggested Tests", language="python")
    b.click(fn=auto_comment_and_test, inputs=code_in, outputs=[commented, tests])
    

In [ ]:
# Launch the UI
demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
